# Gemini Function Calling — Two Functions (Smart Selection)
AI intelligently picks **one or both** functions based on your question.

## Cell 1 — Install & Import

In [ ]:
!pip install google-generativeai -q

import os
import google.generativeai as genai

## Cell 2 — API Key Setup

In [ ]:
GOOGLE_API_KEY = "your-key"   # Replace with your actual key
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

MODEL_NAME = "gemini-2.5-flash"

## Cell 3 — Configure Gemini

In [ ]:
def configure_gemini():
    """Configures the Gemini API."""
    if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your-key":
        print("Error: Please set your GOOGLE_API_KEY.")
        return False
    try:
        genai.configure(api_key=GOOGLE_API_KEY)
        print("Gemini configured successfully.")
        return True
    except Exception as e:
        print(f"Error configuring Gemini API: {e}")
        return False

if not configure_gemini():
    raise SystemExit("Fix API key first.")

## Cell 4 — Define Two Python Functions

- **`calculate`** → arithmetic (add / subtract / multiply / divide)
- **`convert_units`** → unit conversion (km↔miles, kg↔lbs, °C↔°F)

In [ ]:
# ── Function 1 ──────────────────────────────────────────────────────────────
def calculate(operation: str, num1: float, num2: float) -> float:
    """
    Performs arithmetic: add, subtract, multiply, divide.
    """
    op = operation.lower()
    if op == "add":
        return num1 + num2
    elif op == "subtract":
        return num1 - num2
    elif op == "multiply":
        return num1 * num2
    elif op == "divide":
        if num2 == 0:
            return "Error: Division by zero"
        return num1 / num2
    else:
        return f"Invalid operation: {operation}"


# ── Function 2 ──────────────────────────────────────────────────────────────
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """
    Converts between common units.
    Supported pairs:
      km  <-> miles
      kg  <-> lbs
      cm  <-> inches
      celsius <-> fahrenheit
    """
    pair = (from_unit.lower(), to_unit.lower())
    conversions = {
        ("km",       "miles"):      lambda v: v * 0.621371,
        ("miles",    "km"):         lambda v: v * 1.60934,
        ("kg",       "lbs"):        lambda v: v * 2.20462,
        ("lbs",      "kg"):         lambda v: v / 2.20462,
        ("cm",       "inches"):     lambda v: v / 2.54,
        ("inches",   "cm"):         lambda v: v * 2.54,
        ("celsius",  "fahrenheit"): lambda v: (v * 9/5) + 32,
        ("fahrenheit","celsius"):   lambda v: (v - 32) * 5/9,
    }
    if pair in conversions:
        result = conversions[pair](value)
        return f"{value} {from_unit} = {round(result, 4)} {to_unit}"
    else:
        return f"Unsupported conversion: {from_unit} → {to_unit}"


# Quick sanity tests
print(calculate("add", 10, 5))             # 15
print(calculate("multiply", 7, 6))         # 42
print(convert_units(100, "km", "miles"))   # 62.1371 miles
print(convert_units(37, "celsius", "fahrenheit"))  # 98.6 F

## Cell 5 — Define Function Declarations (Gemini Tool Schema)

In [ ]:
calculate_declaration = genai.types.FunctionDeclaration(
    name="calculate",
    description="Performs arithmetic operations: add, subtract, multiply, or divide two numbers.",
    parameters={
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "description": "Operation to perform: 'add', 'subtract', 'multiply', or 'divide'."
            },
            "num1": {"type": "number", "description": "First number."},
            "num2": {"type": "number", "description": "Second number."}
        },
        "required": ["operation", "num1", "num2"]
    }
)

convert_units_declaration = genai.types.FunctionDeclaration(
    name="convert_units",
    description=(
        "Converts a value from one unit to another. "
        "Supported: km/miles, kg/lbs, cm/inches, celsius/fahrenheit."
    ),
    parameters={
        "type": "object",
        "properties": {
            "value":     {"type": "number", "description": "The numeric value to convert."},
            "from_unit": {"type": "string", "description": "Source unit (e.g. 'km', 'celsius', 'kg')."},
            "to_unit":   {"type": "string", "description": "Target unit (e.g. 'miles', 'fahrenheit', 'lbs')."}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
)

print("Function declarations ready.")

## Cell 6 — Create Model + Chat Session

In [ ]:
model = genai.GenerativeModel(
    MODEL_NAME,
    tools=[calculate_declaration, convert_units_declaration]
)
chat_session = model.start_chat()
print("Model ready with both tools.")

## Cell 7 — Function Dispatcher

Routes Gemini's function call to the correct Python function.
Handles **single** or **multiple** function calls in one response.

In [ ]:
FUNCTION_MAP = {
    "calculate":     calculate,
    "convert_units": convert_units,
}

def dispatch(function_name: str, arguments: dict):
    """Calls the correct Python function and returns its result."""
    fn = FUNCTION_MAP.get(function_name)
    if fn is None:
        return f"Unknown function: {function_name}"
    return fn(**arguments)


def extract_function_calls(response):
    """
    Extracts ALL function_call parts from a Gemini response.
    Returns a list of (function_name, arguments) tuples.
    Returns empty list if no function calls (plain text response).
    """
    calls = []
    try:
        parts = response.candidates[0].content.parts
        for part in parts:
            fc = part.function_call
            if fc and fc.name:   # valid function call part
                calls.append((fc.name, dict(fc.args)))
    except (AttributeError, IndexError):
        pass
    return calls


print("Dispatcher ready.")

## Cell 8 — Main Chat Loop

**Try these prompts:**
- `What is 25 + 17?`  → single function (calculate)
- `Convert 100 km to miles`  → single function (convert_units)
- `Add 50 and 30, then convert 100 celsius to fahrenheit`  → **both functions**
- `Multiply 12 by 8 and also convert 75 kg to lbs`  → **both functions**

In [ ]:
def main():
    print("Chat started. Type 'exit' to quit.")
    print("-" * 50)

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "exit":
            print("Exiting...")
            break

        # Send message to Gemini
        response = chat_session.send_message(user_input)

        # Extract any function calls from the response
        calls = extract_function_calls(response)

        if calls:
            # ── AI chose to use function(s) ──────────────────────────────
            if len(calls) == 1:
                print(f"[AI used 1 function]")
            else:
                print(f"[AI used {len(calls)} functions simultaneously]")

            for fn_name, args in calls:
                result = dispatch(fn_name, args)
                print(f"  → {fn_name}({args})  =  {result}")

        else:
            # ── AI replied with plain text (no function needed) ──────────
            print(f"Gemini: {response.text}")

        print()


main()